# The elastic sphere, solved symbolically

The classical building block of micromechanics: an isotropic sphere — or a
stack of concentric isotropic layers — under a uniform remote strain. Because
the material is isotropic, the problem splits by the symmetry of the remote
loading, and each part reduces to an ordinary differential equation in the
radius that SymPy solves in closed form.

Everything here is produced by the differential operators of
Curvilinear differential calculus: `SYMGRAD` gives the
strain, `DIV` the equilibrium equation. Nothing is transcribed from a
textbook. Background: [mura1987](@cite).

In [1]:
using TensND
using LinearAlgebra
using SymPy

Spherical = coorsys_spherical()
θ, ϕ, r = getcoords(Spherical)
𝐞ᶿ, 𝐞ᵠ, 𝐞ʳ = unitvec(Spherical)
@set_coorsys Spherical

𝐞₁, 𝐞₂, 𝐞₃ = unitvec(coorsys_cartesian())
𝕀, 𝕁, 𝕂 = iso_projectors(Val(3), Val(Sym))
𝟏 = tens_Id2(Val(3), Val(Sym))

k, μ = symbols("k μ", positive = true)
ℂ = 3k * 𝕁 + 2μ * 𝕂

3×3×3×3 TensISO{4, 3, Sym{PyCall.PyObject}, 2}:
[:, :, 1, 1] =
 k + 4*μ/3          0          0
         0  k - 2*μ/3          0
         0          0  k - 2*μ/3

[:, :, 2, 1] =
 0  μ  0
 μ  0  0
 0  0  0

[:, :, 3, 1] =
 0  0  μ
 0  0  0
 μ  0  0

[:, :, 1, 2] =
 0  μ  0
 μ  0  0
 0  0  0

[:, :, 2, 2] =
 k - 2*μ/3          0          0
         0  k + 4*μ/3          0
         0          0  k - 2*μ/3

[:, :, 3, 2] =
 0  0  0
 0  0  μ
 0  μ  0

[:, :, 1, 3] =
 0  0  μ
 0  0  0
 μ  0  0

[:, :, 2, 3] =
 0  0  0
 0  0  μ
 0  μ  0

[:, :, 3, 3] =
 k - 2*μ/3          0          0
         0  k - 2*μ/3          0
         0          0  k + 4*μ/3

## The hydrostatic problem

For a remote strain $\mathbb{E}^\infty\propto\boldsymbol{1}$ the
displacement is purely radial, $\underline{u}=u(r)\,\underline{e}^r$.

In [2]:
u = SymFunction("u", real = true)
𝐮 = u(r) * 𝐞ʳ

3-element TensND.TensRotated{1, 3, Sym{PyCall.PyObject}, Tensors.Vec{3, Sym{PyCall.PyObject}}}:
    0
    0
 u(r)

The strain follows from `SYMGRAD`, the stress from the constitutive law, and
the equilibrium equation from `DIV`:

In [3]:
𝛆 = SYMGRAD(𝐮)
𝛔 = ℂ ⊡ 𝛆
𝐓 = 𝛔 ⋅ 𝐞ʳ
eq = factor(simplify(DIV(𝛔) ⋅ 𝐞ʳ))

            ⎛    2                               ⎞
            ⎜ 2 d               d                ⎟
(3⋅k + 4⋅μ)⋅⎜r ⋅───(u(r)) + 2⋅r⋅──(u(r)) - 2⋅u(r)⎟
            ⎜     2             dr               ⎟
            ⎝   dr                               ⎠
──────────────────────────────────────────────────
                          2                       
                       3⋅r                        

A second-order Euler equation in $r$, solved directly:

In [4]:
sol = dsolve(eq, u(r))
û = sol.rhs()

C₁       
── + C₂⋅r
 2       
r        

The two exponents are $r$ and $r^{-2}$, i.e. the familiar
$u(r)=C_1r+C_2/r^2$. The radial traction carries the interface and boundary
conditions of a layered assemblage:

In [5]:
T̂ = tsimplify(tsimplify(subs(𝐓 ⋅ 𝐞ʳ, u(r) => û)))

  4⋅C₁⋅μ         
- ────── + 3⋅C₂⋅k
     3           
    r            

A solid sphere is regular at the origin, so the $r^{-2}$ constant vanishes
and the remaining term is $3k$ times the uniform strain — a state of uniform
hydrostatic stress, as it must be.

## The deviatoric axisymmetric problem

For $\mathbb{E}^\infty=\boldsymbol{1}-3\,\underline{e}_3\otimes\underline{e}_3$
the angular dependence is fixed by the loading and only the radial profiles
remain unknown. The angular functions are generated from the remote strain
itself:

In [6]:
remote_angle_functions(𝐄) = let fʳ = simplify(𝐞ʳ ⋅ 𝐄 ⋅ 𝐞ʳ)
    (diff(fʳ, θ) / 2, diff(fʳ, ϕ) / (2sin(θ)), fʳ)
end

uᶿ = SymFunction("uᶿ", real = true)
uᵠ = SymFunction("uᵠ", real = true)
uʳ = SymFunction("uʳ", real = true)
α, Λ = symbols("α Λ", real = true)

fᶿ, _, fʳ = remote_angle_functions(𝟏 - 3𝐞₃ ⊗ 𝐞₃)
(fᶿ, fʳ)

(3*sin(θ)*cos(θ), 3*sin(θ)^2 - 2)

In [7]:
𝐮ᵈ = uᶿ(r) * fᶿ * 𝐞ᶿ + uʳ(r) * fʳ * 𝐞ʳ
𝛔ᵈ = ℂ ⊡ SYMGRAD(𝐮ᵈ)
𝐓ᵈ = 𝛔ᵈ ⋅ 𝐞ʳ
div𝛔ᵈ = DIV(𝛔ᵈ)

3-element TensND.TensRotated{1, 3, Sym{PyCall.PyObject}, Tensors.Vec{3, Sym{PyCall.PyObject}}}:
                                                                                                                                               r*(4*μ*(3*sin(θ)*cos(θ)*Derivative(uᶿ(r), r)/2 + (6*uʳ(r)*sin(θ)*cos(θ) - 3*uᶿ(r)*sin(θ)*cos(θ))/(2*r))/r^2 + (2*μ*(6*uʳ(r)*sin(θ)*cos(θ)/r - 12*uᶿ(r)*sin(θ)*cos(θ)/r) + (3*k - 2*μ)*(6*sin(θ)*cos(θ)*Derivative(uʳ(r), r) + 3*(-tan(θ)^2 - 1)*uᶿ(r)*sin(θ)*cos(θ)/(r*tan(θ)^2) + 12*uʳ(r)*sin(θ)*cos(θ)/r - 3*uᶿ(r)*sin(θ)^2/(r*tan(θ)) - 12*uᶿ(r)*sin(θ)*cos(θ)/r + 3*uᶿ(r)*cos(θ)^2/(r*tan(θ)))/3)/r^2) + r*(2*μ*(3*sin(θ)*cos(θ)*Derivative(uᶿ(r), r)/2 + (6*uʳ(r)*sin(θ)*cos(θ) - 3*uᶿ(r)*sin(θ)*cos(θ))/(2*r))/r^2 - (2*μ*((3*sin(θ)^2 - 2)*uʳ(r)/r + 3*uᶿ(r)*sin(θ)*cos(θ)/(r*tan(θ))) + (3*k - 2*μ)*((3*sin(θ)^2 - 2)*Derivative(uʳ(r), r) + 2*(3*sin(θ)^2 - 2)*uʳ(r)/r - 3*uᶿ(r)*sin(θ)^2/r + 3*uᶿ(r)*sin(θ)*cos(θ)/(r*tan(θ)) + 3*uᶿ(r)*cos(θ)^2/r)/3)*sin(2*θ)/(2*r^2*sin(θ)^

The angular dependence factors out exactly: dividing by $f^\theta$ and
$f^r$ leaves two coupled radial equations.

In [8]:
eqᶿ = tsimplify(div𝛔ᵈ ⋅ 𝐞ᶿ / fᶿ)
eqʳ = tsimplify(div𝛔ᵈ ⋅ 𝐞ʳ / fʳ)

        2                                                                      ↪
     2 d                  d                 d                                  ↪
3⋅k⋅r ⋅───(uʳ(r)) + 6⋅k⋅r⋅──(uʳ(r)) - 9⋅k⋅r⋅──(uᶿ(r)) - 6⋅k⋅uʳ(r) + 9⋅k⋅uᶿ(r)  ↪
         2                dr                dr                                 ↪
       dr                                                                      ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                               ↪
                                                                             3 ↪

↪           2                                                                  ↪
↪      2   d                  d                 d                              ↪
↪ + 4⋅r ⋅μ⋅───(uʳ(r)) + 8⋅r⋅μ⋅──(uʳ(r)) - 3⋅r⋅μ⋅──(uᶿ(r)) - 26⋅μ⋅uʳ(r) + 21⋅μ⋅ ↪
↪            2                dr                dr                             ↪
↪          dr              

## The Lamé exponents

Substituting the power ansatz $u^\theta=r^\alpha$, $u^r=\Lambda r^\alpha$
turns the differential system into an algebraic one for $(\alpha,\Lambda)$:

In [9]:
eqs = tsimplify.(subs.([eqᶿ, eqʳ], uᶿ(r) => r^α, uʳ(r) => Λ * r^α))
αΛ = solve([e.doit() for e in eqs], [α, Λ])

4-element Vector{Tuple{Sym{PyCall.PyObject}, Sym{PyCall.PyObject}}}:
 (-4, -3/2)
 (1, 1)
 (-2, 3*(k + μ)/(2*μ))
 (3, 3*(3*k - 2*μ)/(15*k + 11*μ))

Four solutions — the four exponents of the deviatoric problem. Two are regular
at the origin and two at infinity, which is exactly what a layered assemblage
needs: two constants per layer, fixed by continuity at each interface.

In [10]:
[(pair[1], simplify(pair[2])) for pair in αΛ]

4-element Vector{Tuple{Sym{PyCall.PyObject}, Sym{PyCall.PyObject}}}:
 (-4, -3/2)
 (1, 1)
 (-2, 3*(k + μ)/(2*μ))
 (3, 3*(3*k - 2*μ)/(15*k + 11*μ))

The general solution is their combination:

In [11]:
ûᶿ = sum(Sym("C$(i + 2)") * r^αΛ[i][1] for i in 1:length(αΛ))
ûʳ = sum(Sym("C$(i + 2)") * αΛ[i][2] * r^αΛ[i][1] for i in 1:length(αΛ))
(ûᶿ, ûʳ)

(C3/r^4 + C4*r + C5/r^2 + C6*r^3, -3*C3/(2*r^4) + C4*r + 3*C5*(k + μ)/(2*r^2*μ) + 3*C6*r^3*(3*k - 2*μ)/(15*k + 11*μ))

and the tractions that carry the interface conditions:

In [12]:
T̂ᶿ = tsimplify(tsimplify(subs(simplify(𝐓ᵈ ⋅ 𝐞ᶿ / fᶿ), uᶿ(r) => ûᶿ, uʳ(r) => ûʳ)))

                     2            5            5  2          2  2            2 ↪
-120⋅C₃⋅k⋅μ - 88⋅C₃⋅μ  + 30⋅C₄⋅k⋅r ⋅μ + 22⋅C₄⋅r ⋅μ  + 45⋅C₅⋅k ⋅r  + 33⋅C₅⋅k⋅r  ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                               5                               ↪
                                              r ⋅(15⋅k + 11⋅μ)                 ↪

↪               7            7  2
↪ ⋅μ + 48⋅C₆⋅k⋅r ⋅μ + 10⋅C₆⋅r ⋅μ 
↪ ───────────────────────────────
↪                                
↪                                

In [13]:
T̂ʳ = tsimplify(tsimplify(subs(simplify(𝐓ᵈ ⋅ 𝐞ʳ / fʳ), uᶿ(r) => ûᶿ, uʳ(r) => ûʳ)))

                     2            5            5  2           2  2             ↪
180⋅C₃⋅k⋅μ + 132⋅C₃⋅μ  + 30⋅C₄⋅k⋅r ⋅μ + 22⋅C₄⋅r ⋅μ  - 135⋅C₅⋅k ⋅r  - 159⋅C₅⋅k⋅ ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                      5                        ↪
                                                     r ⋅(15⋅k + 11⋅μ)          ↪

↪  2            2  2           7           7  2
↪ r ⋅μ - 44⋅C₅⋅r ⋅μ  - 9⋅C₆⋅k⋅r ⋅μ + 6⋅C₆⋅r ⋅μ 
↪ ─────────────────────────────────────────────
↪                                              
↪                                              

## Pure shear gives the same exponents

A different deviatoric loading,
$\mathbb{E}^\infty=\underline{e}_1\otimes\underline{e}_1
-\underline{e}_2\otimes\underline{e}_2$, now with an azimuthal component.
Isotropy demands that it produce the *same* radial exponents; only the angular
functions differ.

In [14]:
fᶿ₂, fᵠ₂, fʳ₂ = remote_angle_functions(𝐞₁ ⊗ 𝐞₁ - 𝐞₂ ⊗ 𝐞₂)
(fᶿ₂, fᵠ₂, fʳ₂)

(sin(θ)*cos(θ)*cos(2*ϕ), -sin(θ)*sin(2*ϕ), sin(θ)^2*cos(2*ϕ))

In [15]:
𝐮ˢ = uᶿ(r) * fᶿ₂ * 𝐞ᶿ + uᵠ(r) * fᵠ₂ * 𝐞ᵠ + uʳ(r) * fʳ₂ * 𝐞ʳ
div𝛔ˢ = DIV(ℂ ⊡ SYMGRAD(𝐮ˢ))

eqᶿˢ = tsimplify(div𝛔ˢ ⋅ 𝐞ᶿ / fᶿ₂)
eqᵠˢ = tsimplify(div𝛔ˢ ⋅ 𝐞ᵠ / fᵠ₂)
eqʳˢ = tsimplify(div𝛔ˢ ⋅ 𝐞ʳ / fʳ₂)

                                                              d                ↪
        2                                               6⋅k⋅r⋅──(uᵠ(r))   6⋅k⋅ ↪
     2 d                  d                 d                 dr               ↪
3⋅k⋅r ⋅───(uʳ(r)) + 6⋅k⋅r⋅──(uʳ(r)) - 9⋅k⋅r⋅──(uᶿ(r)) - ─────────────── + ──── ↪
         2                dr                dr                 2               ↪
       dr                                                   sin (θ)            ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                               ↪
                                                                               ↪

↪   d                                                                          ↪
↪ r⋅──(uᶿ(r))                                                           2      ↪
↪   dr                      6⋅k⋅uᵠ(r)               6⋅k⋅uᶿ(r)      2   d       ↪
↪ ─────────── - 6⋅k⋅uʳ(r) +

The azimuthal equation forces $u^\varphi=u^\theta$: the two transverse
profiles are not independent.

In [16]:
X = symbols("X", real = true)
uᵠsol = solve(tsimplify(diff(subs(eqᵠˢ, sin(θ)^2 => 1 / X), X)), uᵠ(r))[1]

uᶿ(r)

With that identification the remaining system reproduces the same exponents as
the axisymmetric case:

In [17]:
eqs₂ = tsimplify.(subs.([eqᶿˢ, eqʳˢ], uᵠ(r) => r^α, uᶿ(r) => r^α, uʳ(r) => Λ * r^α))
αΛ₂ = solve([e.doit() for e in eqs₂], [α, Λ])

sort(string.(first.(αΛ))) == sort(string.(first.(αΛ₂)))

true

The deviatoric response of an isotropic sphere therefore depends on the
*symmetry class* of the remote loading, not on its particular orientation —
which is what lets an $N$-layer assemblage be solved once and reused for any
deviatoric loading.

## Assembling $N$ layers

In a stack of concentric layers, layer $i$ occupies
$R_{i-1}\le r\le R_i$ with its own moduli $(k_i,\mu_i)$ and carries the
constants found above. Continuity of the displacement and of the radial
traction at each interface gives two scalar equations per interface in the
hydrostatic problem and four in the deviatoric one; regularity at the center
and the remote condition close the system.

The expressions `T̂`, `T̂ᶿ` and `T̂ʳ` derived above are exactly the quantities
to be matched, so the assembly is ordinary linear algebra on the constants —
no further tensor calculus is required.

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*